In [1]:
import pandas as pd

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek

    # Lag features: use only past data
    df["lag_1"] = df["load"].shift(1)
    df["lag_24"] = df["load"].shift(24)

    # Rolling mean over previous 24 hours, excluding current row
    df["rolling_mean_24"] = df["load"].shift(1).rolling(window=24).mean()

    return df

In [2]:
# Path to CSV (expects columns: timestamp, load)
import os
DATA_DIR = "data_timeseries"
CSV_PATH = os.path.join(DATA_DIR, "load_hourly.csv")

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
df.head(10)

Loaded 144 rows


,timestamp,load
0,2024-01-01 00:00:00,58.2
1,2024-01-01 01:00:00,52.1
2,2024-01-01 02:00:00,49.8
3,2024-01-01 03:00:00,48.5
4,2024-01-01 04:00:00,50.2
5,2024-01-01 05:00:00,54.3
6,2024-01-01 06:00:00,62.1
7,2024-01-01 07:00:00,78.4
8,2024-01-01 08:00:00,88.2
9,2024-01-01 09:00:00,92.5


In [3]:
# Build features and show result
df_feat = make_features(df)
df_feat.head(30)

,timestamp,load,hour,day_of_week,lag_1,lag_24,rolling_mean_24
0,2024-01-01 00:00:00,58.2,0,0,NaN,NaN,NaN
1,2024-01-01 01:00:00,52.1,1,0,58.2,NaN,NaN
2,2024-01-01 02:00:00,49.8,2,0,52.1,NaN,NaN
3,2024-01-01 03:00:00,48.5,3,0,49.8,NaN,NaN
4,2024-01-01 04:00:00,50.2,4,0,48.5,NaN,NaN
5,2024-01-01 05:00:00,54.3,5,0,50.2,NaN,NaN
6,2024-01-01 06:00:00,62.1,6,0,54.3,NaN,NaN
7,2024-01-01 07:00:00,78.4,7,0,62.1,NaN,NaN
8,2024-01-01 08:00:00,88.2,8,0,78.4,NaN,NaN
9,2024-01-01 09:00:00,92.5,9,0,88.2,NaN,NaN


In [4]:
# After row 24, lag_24 and rolling_mean_24 are filled
df_feat[["timestamp", "load", "lag_1", "lag_24", "rolling_mean_24"]].iloc[20:30]

,timestamp,load,lag_1,lag_24,rolling_mean_24
20,2024-01-01 20:00:00,72.3,75.8,NaN,NaN
21,2024-01-01 21:00:00,66.5,72.3,NaN,NaN
22,2024-01-01 22:00:00,62.8,66.5,NaN,NaN
23,2024-01-01 23:00:00,59.4,62.8,NaN,NaN
24,2024-01-02 00:00:00,57.8,59.4,58.2,73.641667
25,2024-01-02 01:00:00,53.2,57.8,52.1,73.625000
26,2024-01-02 02:00:00,50.1,53.2,49.8,73.670833
27,2024-01-02 03:00:00,49.2,50.1,48.5,73.683333
28,2024-01-02 04:00:00,51.0,49.2,50.2,73.712500
29,2024-01-02 05:00:00,55.6,51.0,54.3,73.745833


In [5]:
def time_split(df: pd.DataFrame):
    df = df.sort_values("timestamp").reset_index(drop=True)

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df

In [ ]:
def rolling_origin_splits(df: pd.DataFrame, train_size: int, val_size: int, step: int):
    df = df.sort_values("timestamp").reset_index(drop=True)
    splits = []

    start = 0
    while start + train_size + val_size <= len(df):
        train_idx = range(start, start + train_size)
        val_idx = range(start + train_size, start + train_size + val_size)
        splits.append((train_idx, val_idx))
        start += step

    return splits

In [ ]:
def seasonal_naive_forecast(history, horizon, season_length=24):
    if len(history) < season_length:
        raise ValueError("Not enough history for seasonal naive forecast")

    forecast = []
    for h in range(horizon):
        forecast.append(history[-season_length + (h % season_length)])
    return forecast

In [ ]:
history = list(range(100))  # pretend these are loads
pred = seasonal_naive_forecast(history, horizon=5, season_length=24)
print(pred)